In [1]:
import os
import sys
import logging
import random
import numpy as np

# Keep Kaggle output readable while TensorFlow initializes.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)
logging.getLogger("tensorflow").setLevel(logging.FATAL)

import tensorflow as tf

# Fixed seeds make the comparison as reproducible as practical.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


In [2]:
# REPOSITORY CONFIGURATION

REPO_NAME = "RefraScan"
GITHUB_USER = "KyziaPi"
BRANCH_NAME = "Model-Experiment"  # <-- point this at your branch

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
REPO_PATH = os.path.join("/kaggle/working", REPO_NAME)

if not os.path.exists(REPO_PATH):
    print(f"Cloning {REPO_NAME} branch '{BRANCH_NAME}'...")
    !env GIT_TERMINAL_PROMPT=0 git clone -b {BRANCH_NAME} {REPO_URL}
else:
    print(f"{REPO_NAME} already exists. Updating branch '{BRANCH_NAME}'...")
    !cd {REPO_PATH} && env GIT_TERMINAL_PROMPT=0 git fetch --all && git checkout {BRANCH_NAME} && git pull origin {BRANCH_NAME}

if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

import sys
import math
import pandas as pd
from tensorflow.keras.applications.densenet import preprocess_input

from src.preprocessing import (
    load_and_clean_data,
    encode_target,
    validate_dataset,
    majority_class_baseline,
)
from src.cross_validation import run_cross_validation, split_holdout_test
from src.evaluate import majority_class_baseline_metrics

print(f"Environment configured successfully! Working on branch: {BRANCH_NAME}")


Cloning RefraScan branch 'Model-Experiment'...
Cloning into 'RefraScan'...
remote: Enumerating objects: 511, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 511 (delta 103), reused 94 (delta 44), pack-reused 344 (from 1)
Receiving objects: 100% (511/511), 48.69 MiB | 43.70 MiB/s, done.
Resolving deltas: 100% (277/277), done.
Environment configured successfully! Working on branch: Model-Experiment


In [3]:
# DATASET VALIDATION + MAJORITY-CLASS BASELINE

DATASET_DIR = '/kaggle/input/datasets/yerikaelainegueco/fundus-images-with-refractive-values'
CSV_PATH = os.path.join(DATASET_DIR, 'RefraScan_dataset.csv')
IMG_DIR = os.path.join(DATASET_DIR, 'FundusImages')

# Load the original records. No refractive measurement is passed to the model.
df = load_and_clean_data(CSV_PATH, IMG_DIR)

# Validate images, labels, patient grouping, and target-derived columns BEFORE training.
df = validate_dataset(df, patient_col="ID", target_col="classification")

# Encode only the three clinical target classes.
df = encode_target(df)

# Establish the descriptive majority-class baseline on the development pool.
development_df, holdout_df = split_holdout_test(
    df,
    patient_col="ID",
    target_col="classification_encoded",
    test_size=0.15,
)

baseline = majority_class_baseline_metrics(
    development_df["classification_encoded"].values
)

print("\n--- Majority-Class Baseline (Development Data) ---")
print(f"Majority class : {baseline['majority_class_name']}")
print(f"Accuracy       : {baseline['accuracy']:.4f}")
print(f"Balanced Acc.  : {baseline['balanced_accuracy']:.4f}")
print(f"Macro F1       : {baseline['macro_f1']:.4f}")
print("\nDevelopment class distribution:")
print(development_df["classification"].value_counts())
print("\nHoldout class distribution (kept untouched):")
print(holdout_df["classification"].value_counts())



DATASET VALIDATION

Total records       : 1,018
Unique patients     : 517
Missing patient IDs : 0
Missing labels      : 0
Duplicate rows      : 0

Class distribution:
classification
Myopia        660
Hyperopia     236
Emmetropia    122
Name: count, dtype: int64

Patients with multiple target classes: 50
These patients have different classifications between their eyes. This is allowed.

Example mixed-class patients:
 ID classification
  2     Emmetropia
  2         Myopia
  4     Emmetropia
  4      Hyperopia
  7     Emmetropia
  7         Myopia
 14         Myopia
 14     Emmetropia
 19     Emmetropia
 19      Hyperopia
 24      Hyperopia
 24     Emmetropia
 27     Emmetropia
 27         Myopia
 34     Emmetropia
 34      Hyperopia
 42     Emmetropia
 42         Myopia
 44         Myopia
 44      Hyperopia

Valid target classes confirmed:
['Emmetropia', 'Hyperopia', 'Myopia']

Target-derived refractive measurement columns detected:
  - sphere
  - cylinder
  - spherical_equivalent

The

In [4]:
# CONTROLLED IMAGE-ONLY 10-FOLD COMPARISON
# The three architecture notebooks use the same: 
#   * 15% patient-level holdout rule
#   * 85% development data
#   * 10-fold Stratified Group CV
#   * 224x224 input size
#   * image preprocessing and mild augmentation
#   * focal loss + training-fold class weights
#   * primary metrics: Macro F1 and balanced accuracy
#   * secondary metric: accuracy
# IMPORTANT: use_metadata=False means sphere, cylinder, spherical equivalent,
# and age are NOT inputs in this architecture-comparison stage.

results = run_cross_validation(
    df=df,
    model_name="densenet121",
    preprocess_input=preprocess_input,
    patient_col="ID",
    target_col="classification_encoded",
    n_splits=10,
    batch_size=16,
    epochs=30,
    learning_rate=1e-4,
    use_metadata=False,
    output_dir=f"/kaggle/working/{REPO_NAME}/artifacts",
    fine_tune=False,
)

# Save fold metrics so the three notebooks can be compared consistently.
results["fold_results"].to_csv(
    f"/kaggle/working/{REPO_NAME}/densenet121_image_only_cv_results.csv",
    index=False,
)

print("\nImage-only experiment completed for DenseNet121.")



PATIENT-LEVEL HOLDOUT SPLIT
Development records : 864
Holdout records     : 154
Development patients: 439
Holdout patients    : 78
Patient overlap     : 0

The holdout set is now untouched and will not be used for architecture/model selection.

DENSENET121 | IMAGE ONLY
10-FOLD STRATIFIED GROUP CROSS-VALIDATION

--- Fold 1/10 ---


I0000 00:00:1786722998.117909      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786722998.120874      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/30
 2/49 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.0938 - loss: 1.4656  

I0000 00:00:1786723031.636402     141 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 767ms/step - accuracy: 0.3594 - loss: 1.0056
Epoch 1: val_loss improved from None to 0.39845, saving model to /kaggle/working/RefraScan/artifacts/densenet121_image_fold_1_frozen.weights.h5

Epoch 1: finished saving model to /kaggle/working/RefraScan/artifacts/densenet121_image_fold_1_frozen.weights.h5
49/49 ━━━━━━━━━━━━━━━━━━━━ 91s 1s/step - accuracy: 0.4570 - loss: 0.8268 - val_accuracy: 0.6235 - val_loss: 0.3984
Epoch 2/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 374ms/step - accuracy: 0.5796 - loss: 0.5410
Epoch 2: val_loss did not improve from 0.39845
49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 379ms/step - accuracy: 0.5802 - loss: 0.5384 - val_accuracy: 0.2812 - val_loss: 0.6315
Epoch 3/30
 1/49 ━━━━━━━━━━━━━━━━━━━━ 5s 116ms/step - accuracy: 0.5625 - loss: 0.3086

...

49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step - accuracy: 0.6380 - loss: 0.4649
Epoch 3: val_loss did not improve from 0.46387
49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 358ms/step - accuracy: 0.6250 - loss: 0.4816 - val_accuracy: 0.4062 - val_loss: 0.6026
Epoch 4/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step - accuracy: 0.6605 - loss: 0.4002
Epoch 4: val_loss did not improve from 0.46387
49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 327ms/step - accuracy: 0.6701 - loss: 0.3942 - val_accuracy: 0.3438 - val_loss: 0.7813
Epoch 5/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 336ms/step - accuracy: 0.6330 - loss: 0.4395
Epoch 5: val_loss did not improve from 0.46387
49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 340ms/step - accuracy: 0.6456 - loss: 0.4117 - val_accuracy: 0.5312 - val_loss: 0.6885
Epoch 6/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - accuracy: 0.6477 - loss: 0.3944
Epoch 6: val_loss did not improve from 0.46387
49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 326ms/step - accuracy: 0.6598 - loss: 0.4002 - val_accuracy: 0.4688 - val_loss: 0.6239
Epoch 7

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 377ms/step - accuracy: 0.6914 - loss: 0.4454
Epoch 3: val_loss did not improve from 0.40501
49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 381ms/step - accuracy: 0.6564 - loss: 0.4636 - val_accuracy: 0.2188 - val_loss: 0.6090
Epoch 4/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 0.6684 - loss: 0.4055
Epoch 4: val_loss did not improve from 0.40501
49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 371ms/step - accuracy: 0.6346 - loss: 0.4162 - val_accuracy: 0.4062 - val_loss: 0.6580
Epoch 5/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 355ms/step - accuracy: 0.6478 - loss: 0.4164
Epoch 5: val_loss did not improve from 0.40501
49/49 ━━━━━━━━━━━━━━━━━━━━ 17s 359ms/step - accuracy: 0.6359 - loss: 0.4200 - val_accuracy: 0.3125 - val_loss: 0.6212
Epoch 6/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step - accuracy: 0.6343 - loss: 0.4051
Epoch 6: val_loss did not improve from 0.40501
49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 371ms/step - accuracy: 0.6346 - loss: 0.4065 - val_accuracy: 0.3750 - val_loss: 0.6810
Epoch 7

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - accuracy: 0.6233 - loss: 0.5348
Epoch 3: val_loss did not improve from 0.35552
49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 323ms/step - accuracy: 0.6361 - loss: 0.4967 - val_accuracy: 0.2500 - val_loss: 0.6979
Epoch 4/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 328ms/step - accuracy: 0.6387 - loss: 0.4479
Epoch 4: val_loss did not improve from 0.35552
49/49 ━━━━━━━━━━━━━━━━━━━━ 16s 332ms/step - accuracy: 0.6477 - loss: 0.4286 - val_accuracy: 0.2500 - val_loss: 0.6797
Epoch 5/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 368ms/step - accuracy: 0.6637 - loss: 0.3988
Epoch 5: val_loss did not improve from 0.35552
49/49 ━━━━━━━━━━━━━━━━━━━━ 18s 372ms/step - accuracy: 0.6658 - loss: 0.3944 - val_accuracy: 0.2500 - val_loss: 0.6847
Epoch 6/30
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 357ms/step - accuracy: 0.6692 - loss: 0.3768
Epoch 6: val_loss did not improve from 0.35552
49/49 ━━━━━━━━━━━━━━━━━━━━ 17s 361ms/step - accuracy: 0.6761 - loss: 0.3779 - val_accuracy: 0.2500 - val_loss: 0.6987
Epoch 7